# 📘 Notebook 2: Tools and Agents

## What You'll Learn
- What are tools in LangChain
- Creating custom tools
- Building your first agent
- Understanding ReAct agents
- Tool calling with LLMs

In [ ]:
!pip install langchain langchain-core langchain-openai langgraph -q

In [ ]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")

## Step 1: What Are Tools?

**Tools** are functions that AI agents can call to perform actions beyond text generation.

In [ ]:
from langchain.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information. Use this for current events or facts."""
    # Simulated search - in production use actual API
    return f"Search results for '{query}': Found relevant information online."

@tool
def calculator(expression: str) -> float:
    """Calculate mathematical expressions. Use for any math problems."""
    try:
        return eval(expression)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_time() -> str:
    """Get the current time."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M:%S")

print("Tools created!")
print(f"Available tools: {[t.name for t in [search_web, calculator, get_time]]}")

### 🎯 Exercise 1
Create a tool that converts temperatures between Celsius and Fahrenheit.

In [ ]:
# YOUR CODE HERE
@tool
def convert_temperature(value: float, unit: str) -> float:
    """Convert temperature between Celsius and Fahrenheit."""
    if unit.lower() == 'celsius':
        return (value * 9/5) + 32  # C to F
    else:
        return (value - 32) * 5/9  # F to C

# Test it
print(f"32°F = {convert_temperature.invoke({'value': 32, 'unit': 'fahrenheit'}})°C")
print(f"100°C = {convert_temperature.invoke({'value': 100, 'unit': 'celsius'})}°F")

## Step 2: Create Your First Agent

An **agent** is an AI that can decide which tools to use automatically.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# Initialize LLM
llm = ChatOpenAI(model="gpt-3.5-turbo")

# List of tools
tools = [search_web, calculator, get_time]

# Create ReAct agent
agent = create_react_agent(llm, tools)

print("Agent created successfully! 🤖")

## Step 3: Run the Agent

Watch how the agent decides which tools to use!

In [ ]:
# Simple question - no tool needed
print("=" * 60)
print("Question 1: Simple greeting")
print("=" * 60)

for chunk in agent.stream({"messages": [("human", "Hi! How are you?")]}):
    print(chunk)

In [ ]:
# Math question - should use calculator
print("\n" + "=" * 60)
print("Question 2: Math problem")
print("=" * 60)

for chunk in agent.stream({"messages": [("human", "What is 123 * 456 + 789?")]}):
    print(chunk)

In [ ]:
# Time question - should use get_time tool
print("\n" + "=" * 60)
print("Question 3: Current time")
print("=" * 60)

for chunk in agent.stream({"messages": [("human", "What time is it right now?")]}):
    print(chunk)

### 🎯 Exercise 2
Ask the agent a multi-step question that requires multiple tool uses.

In [ ]:
# YOUR TURN - Try a complex question!
complex_question = "What is the current time, and what is 25% of 480?"

print(f"Asking: {complex_question}")
print("=" * 60)

for chunk in agent.stream({"messages": [("human", complex_question)]}):
    print(chunk)

## Step 4: Custom Agent with Personality

Let's customize our agent with a specific persona.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Custom system prompt
custom_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a friendly science teacher who loves making learning fun!
    - Explain things in simple terms
    - Use analogies when possible
    - Be encouraging and enthusiastic
    - Use emojis to make it engaging 🌟"""),
    ("human", "{input}"),
])

# Create agent with custom prompt
teacher_agent = create_react_agent(llm, tools, prompt=custom_prompt)

# Test it
question = "Why is the sky blue?"
print(f"Asking teacher agent: {question}\n")

for chunk in teacher_agent.stream({"messages": [("human", question)]}):
    print(chunk)

## Step 5: Tool Calling Agent (Modern Approach)

Newer LLMs have native tool calling capabilities.

In [ ]:
from langgraph.prebuilt import create_tool_calling_agent

# Create tool calling agent
tool_agent = create_tool_calling_agent(llm, tools)

# Test it
print("Testing tool calling agent:")
print("=" * 60)

for chunk in tool_agent.stream({"messages": [("human", "Calculate 15 * 23 and tell me the time")]}):
    print(chunk)

## 📝 Summary

✅ Created custom tools with @tool decorator
✅ Built a ReAct agent that uses tools
✅ Observed agent decision-making
✅ Customized agent personality
✅ Used modern tool calling agents

## 🏆 Final Challenge

Build a "Homework Helper" agent with these tools:
- Calculator for math
- Spell checker
- Fact verifier

Test it with homework questions!

In [ ]:
# YOUR CODE HERE - Build Homework Helper!

@tool
def spell_check(text: str) -> str:
    """Check spelling in text."""
    # Simplified - just echo back
    return f"Text reviewed: '{text}' (No errors found in this demo)"

@tool
def verify_fact(claim: str) -> str:
    """Verify if a fact is correct."""
    return f"Fact check for '{claim}': This appears to be accurate based on general knowledge."

homework_tools = [calculator, spell_check, verify_fact]
homework_agent = create_react_agent(llm, homework_tools)

# Test with homework question
hw_question = "Is it true that water boils at 100°C? Also calculate 15% of 200."
print(f"Homework Helper:\n{hw_question}\n")
print("=" * 60)

for chunk in homework_agent.stream({"messages": [("human", hw_question)]}):
    print(chunk)

---
**Great job!** Ready for Notebook 3 - RAG Systems! 🚀